# Trigonometric functions

This notebook covers `es.math.sin`, `cos`, `tan` and `sinc`. They take a `Parameter` or a
`DescriptorNumber` and return a new one with the value, unit, variance and bounds worked out: the variance
to first order, the bounds as the exact range of the function over the bounds of the input.

The examples are taken from a diffraction experiment (Bragg's law), but each one stands alone. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## Angles need a unit

The argument of `sin`, `cos` and `tan` must have an angle unit, `rad` or `deg`, and the result is
dimensionless. A length is refused:

In [ ]:
try:
    es.math.sin(Parameter(name='v', value=1.0, unit='m', variance=0.1**2))
except Exception as error:
    print(f'{type(error).__name__}: {error}')

## The sine and Bragg's law

Bragg's law relates the spacing $d$ of the lattice planes to the scattering angle $\theta$ and the
wavelength $\lambda$, and the scattering vector is $q$:

$$
d = \frac{\lambda}{2\sin\theta}, \qquad q = \frac{4\pi\sin\theta}{\lambda}.
$$

The uncertainty is converted correctly: $\sin 15^\circ$ has an error of $\cos 15^\circ \times \sigma_\theta$
with $\sigma_\theta$ in radians. The wavelength is a measured constant, so it is a `DescriptorNumber` (no
bounds); the angle may be fitted, so it is a `Parameter`:

In [ ]:
theta = Parameter(name='theta', value=15.0, unit='deg', variance=0.1**2, min=5.0, max=60.0)
wavelength = DescriptorNumber(name='wavelength', value=1.5406, unit='angstrom', variance=0.0005**2)

s = es.math.sin(theta)
print(show(s))

Ordinary arithmetic then gives $d$ and $q$, with their uncertainties and bounds:

In [ ]:
d = wavelength / (2 * s)
q = 4 * np.pi * s / wavelength
print(show(d))
print(show(q))

### Exercise 1: Scattering vector

X-rays of wavelength 1.54 Å (known exactly) are scattered at $\theta = 22.5 \pm 0.2^\circ$. The scattering vector is $q = 4\pi\sin\theta/\lambda$. What is $q$?

In [ ]:
theta = Parameter(name='theta', value=22.5, variance=0.2**2, unit='deg')
wavelength = DescriptorNumber(name='wavelength', value=1.54, variance=0.0, unit='angstrom')

<details class="question">
<summary>Hint 1</summary>

The angle has the unit `deg`, which `es.math.sin` understands; the result is dimensionless.

</details>

<details class="question">
<summary>Hint 2</summary>

$4\pi$ is a plain number, `4 * np.pi`. Divide by the wavelength last, so that the unit becomes 1/Å.

</details>

In [ ]:
%%solution The bounds are those of the sine, $\pm 1$, scaled by $4\pi/\lambda$.
q = 4 * np.pi * es.math.sin(theta) / wavelength
print(show(q))

### Exercise 2: Lattice spacing

For the same measurement, $\theta = 22.5 \pm 0.2^\circ$ and $\lambda = 1.54$ Å, what is the lattice spacing $d = \lambda/(2\sin\theta)$?

In [ ]:
theta = Parameter(name='theta', value=22.5, variance=0.2**2, unit='deg')
wavelength = DescriptorNumber(name='wavelength', value=1.54, variance=0.0, unit='angstrom')

<details class="question">
<summary>Hint 1</summary>

Compute the sine first, then divide the wavelength by twice the sine.

</details>

<details class="question">
<summary>Hint 2</summary>

A `DescriptorNumber` and a `Parameter` can be combined; the result is a `Parameter`.

</details>

In [ ]:
%%solution
d = wavelength / (2 * es.math.sin(theta))
print(show(d))

## The cosine and the tangent

The same rules apply. The tangent of an angle whose bounds reach $90^\circ$ has no upper limit:

In [ ]:
angle = Parameter(name='angle', value=30.0, unit='deg', variance=1.0**2, min=0.0, max=90.0)

for function in (es.math.cos, es.math.tan):
    print(f'{function.__name__:4}', show(function(angle)))

A flat detector at a distance $L$ from the sample shows a beam scattered by an angle $2\theta$ at a height
$y = L\tan 2\theta$ above the direct beam.

### Exercise 3: Position of a spot

A flat detector is 1.50 ± 0.01 m from the sample. A beam scattered by $2\theta = 30.0 \pm 0.2^\circ$ hits it at a height $y = L\tan(2\theta)$ above the direct beam. What is $y$?

In [ ]:
distance = Parameter(name='distance', value=1.50, variance=0.01**2, unit='m')
two_theta = Parameter(name='two_theta', value=30.0, variance=0.2**2, unit='deg')

<details class="question">
<summary>Hint 1</summary>

`es.math.tan` accepts the angle in degrees.

</details>

<details class="question">
<summary>Hint 2</summary>

A length times a dimensionless number is a length.

</details>

In [ ]:
%%solution
y = distance * es.math.tan(two_theta)
print(show(y))

## The sinc function

`sinc(x)` is $\sin(x)/x$, not the normalised $\sin(\pi x)/(\pi x)$ of NumPy. It takes radians or a
dimensionless number, and refuses degrees. The intensity of single-slit diffraction is proportional to
$\mathrm{sinc}^2$:

In [ ]:
x = Parameter(name='x', value=1.2, variance=0.05**2, min=0.5, max=3.0)
print(show(es.math.sinc(x) ** 2))

try:
    es.math.sinc(Parameter(name='v', value=1.0, unit='deg', variance=0.1**2))
except Exception as error:
    print(f'{type(error).__name__}: {error}')

### Exercise 4: Single-slit intensity

The intensity of single-slit diffraction, relative to the central maximum, is $\mathrm{sinc}^2(x)$. What is it at $x = 2.0 \pm 0.1$?

In [ ]:
x = Parameter(name='x', value=2.0, variance=0.1**2)

<details class="question">
<summary>Hint 1</summary>

`sinc(x)` is $\sin(x)/x$; it takes a dimensionless number or radians.

</details>

<details class="question">
<summary>Hint 2</summary>

Square the result with `**`.

</details>

In [ ]:
%%solution The intensity lies between 0 and 1: $\mathrm{sinc}$ is at most 1, and a square is never negative.
intensity = es.math.sinc(x) ** 2
print(show(intensity))

## Exact bounds

The bounds of the input carry through. The sine of an angle that may vary between $0^\circ$ and $180^\circ$
reaches 1 in the middle, so the upper bound of the result is exactly 1, not the sine of either end:

In [ ]:
wide = Parameter(name='wide', value=80.0, unit='deg', variance=1.0**2, min=0.0, max=180.0)
print(show(es.math.sin(wide)))

## Charts

The lattice spacing $d(\theta)$ of the Bragg example, for an angle with an uncertainty of $1^\circ$ (ten
times larger than above, so that the band can be seen). The band is widest at small angles: the relative
error of $d$ is $\cot\theta\,\sigma_\theta$. The chart is interactive: hover over the curve to read the
values. It loads Plotly from the internet, so it needs a connection. The charts are shown with a small
helper, `show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the
same.

In [ ]:
angles = np.linspace(5.0, 60.0, 60)
wavelength = DescriptorNumber(name='wavelength', value=1.5406, unit='angstrom', variance=0.0005**2)

values, errors = [], []
for value in angles:
    angle = Parameter(name='theta', value=value, unit='deg', variance=1.0**2)
    spacing = wavelength / (2 * es.math.sin(angle))
    values.append(spacing.value)
    errors.append(spacing.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=angles,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=angles,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)

fig.add_trace(
    go.Scatter(
        x=angles,
        y=values,
        mode='lines',
        name='d',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='theta = %{x:.1f} deg<br>d = %{y:.3f} ± %{customdata:.3f} Å<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='theta / deg',
    yaxis_title='d / Å',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.6, y=0.95),
)
show_figure(fig)

The error of the sine itself is $|\cos\theta|\,\sigma_\theta$. Here it is compared with a Monte Carlo
sample of the same sine, for an uncertainty of 0.3 rad:

In [ ]:
sigma = 0.3
angles = np.linspace(0.0, 2 * np.pi, 80)

first_order = np.array([
    es.math.sin(Parameter(name='angle', value=value, unit='rad', variance=sigma**2)).error
    for value in angles
])

rng = np.random.default_rng(seed=1)
samples = rng.normal(angles[:, None], sigma, size=(len(angles), 20_000))
monte_carlo = np.sin(samples).std(axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=angles, y=first_order, mode='lines', name='first order (es.math)'))
fig.add_trace(
    go.Scatter(x=angles, y=monte_carlo, mode='lines', line=dict(dash='dash'), name='Monte Carlo')
)
fig.update_layout(
    xaxis_title='angle / rad',
    yaxis_title='error of sin(angle)',
    margin=dict(l=60, r=20, t=60, b=50),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=1.1),
)
show_figure(fig, height=380)

## Good to know

The two curves agree to within a few per cent away from the extremes. At $\pi/2$ and $3\pi/2$ the sine is
flat, its derivative is 0, and the first-order error drops to (almost) zero, while the Monte Carlo error
stays near $\sigma^2/\sqrt{2}\approx 0.06$:

In [ ]:
top = Parameter(name='top', value=np.pi / 2, unit='rad', variance=0.3**2)
print(es.math.sin(top).error)

rng = np.random.default_rng(seed=2)
print(np.sin(rng.normal(np.pi / 2, 0.3, 200_000)).std())

Close to a pole of the tangent the first-order variance means nothing at all. An angle within one standard
deviation of a pole is refused:

In [ ]:
try:
    es.math.tan(Parameter(name='v', value=np.pi / 2, unit='rad', variance=0.1**2))
except ValueError as error:
    print(error)